# 04 — Decision Analysis: Calibration, Threshold/Cost, Interpretation, Test Evaluation

Goal: turn probabilities into an actionable retention policy, verify the
model is honest about its confidence, explain its decisions, and — **only
once, at the very end** — evaluate on the untouched test set.

> This notebook is the only place the test set is used, and only after the
> threshold has been frozen using validation data alone.


In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

from src.data import TARGET_COLUMN, ID_COLUMN, get_feature_columns, RANDOM_SEED
from src.features import build_full_pipeline
from src.evaluate import (
    evaluate_at_threshold, threshold_search, find_optimal_threshold,
    cost_sensitivity_analysis, permutation_importance_report, shap_explanation,
    bootstrap_metric_ci, subgroup_metrics, risk_group, recommended_action,
    DEFAULT_FN_COST, DEFAULT_FP_COST,
)
from src.predict import ModelArtifact, generate_final_risk_report

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

numerical_columns, categorical_columns = get_feature_columns(train_df)
feature_cols = numerical_columns + categorical_columns

X_train, y_train = train_df[feature_cols], train_df[TARGET_COLUMN]
X_val, y_val = val_df[feature_cols], val_df[TARGET_COLUMN]
X_test, y_test = test_df[feature_cols], test_df[TARGET_COLUMN]

# Re-fit the chosen model family from notebook 03 (logistic regression is the
# usual winner on this schema; swap in whichever model 03_modeling.ipynb selected).
final_pipeline = build_full_pipeline(LogisticRegression(max_iter=1000, C=0.3, random_state=RANDOM_SEED),
                                      numerical_columns, categorical_columns)
final_pipeline.fit(X_train, y_train)
val_proba = final_pipeline.predict_proba(X_val)[:, 1]
print(evaluate_at_threshold(y_val, val_proba))


## 1. Calibration

A model can rank customers correctly (good ROC-AUC) while its raw
probabilities are poorly calibrated (e.g. everyone it calls "80% likely to
churn" actually churns only 50% of the time). Since we use probabilities
directly for cost calculations and risk-group thresholds, calibration
matters.


In [ ]:
prob_true, prob_pred = calibration_curve(y_val, val_proba, n_bins=10, strategy="quantile")
brier = brier_score_loss(y_val, val_proba)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(prob_pred, prob_true, marker="o", label="Model")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed churn rate")
ax.set_title(f"Reliability curve (Brier score = {brier:.4f})")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/calibration_curve.png", dpi=120)
plt.show()


In [ ]:
calibrated = CalibratedClassifierCV(final_pipeline, method="isotonic", cv=5)
calibrated.fit(X_train, y_train)
calibrated_proba_val = calibrated.predict_proba(X_val)[:, 1]
print(f"Uncalibrated Brier: {brier:.4f}")
print(f"Isotonic-calibrated Brier: {brier_score_loss(y_val, calibrated_proba_val):.4f}")


**Did calibration improve probability quality?** Compare the two Brier
scores above (lower is better). **Did it change threshold decisions?** Rerun
the threshold search (next section) on `calibrated_proba_val` instead of
`val_proba` and compare the resulting optimal threshold — a well-ranked but
poorly calibrated model can shift its cost-optimal threshold noticeably
after calibration even if ROC-AUC barely changes, because the *absolute*
probability values (not just their ranking) drive the cost calculation.


## 2. Threshold optimization by expected business cost

In [ ]:
threshold_table = threshold_search(y_val, val_proba, fn_cost=DEFAULT_FN_COST, fp_cost=DEFAULT_FP_COST)
threshold_table.head(10)


In [ ]:
optimal_threshold, _ = find_optimal_threshold(y_val, val_proba, fn_cost=DEFAULT_FN_COST, fp_cost=DEFAULT_FP_COST)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(threshold_table.sort_values("threshold")["threshold"],
          threshold_table.sort_values("threshold")["expected_cost"], color="#C44E52", label="Expected cost")
ax1.axvline(optimal_threshold, linestyle="--", color="black", label=f"Optimal threshold = {optimal_threshold:.2f}")
ax1.set_xlabel("Decision threshold")
ax1.set_ylabel("Expected cost ($)")
ax1.set_title(f"Cost curve — FN cost=${DEFAULT_FN_COST:.0f}, FP cost=${DEFAULT_FP_COST:.0f}")
ax1.legend()
plt.tight_layout()
plt.savefig("../reports/figures/cost_curve.png", dpi=120)
plt.show()

print(f"Frozen operating threshold: {optimal_threshold:.2f}")


**How sensitive is the threshold to the assumed cost ratio?** See the
sensitivity table below — the threshold moves substantially as the assumed
false-negative cost changes, which is exactly why we don't hard-code a
threshold assumption without stating it explicitly.


In [ ]:
sensitivity_table = cost_sensitivity_analysis(y_val, val_proba, fn_costs=[200, 500, 1000, 2000])
sensitivity_table


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(sensitivity_table["fn_cost_assumption"], sensitivity_table["optimal_threshold"], marker="o")
ax.set_xlabel("Assumed false-negative cost ($)")
ax.set_ylabel("Optimal threshold")
ax.set_title("Threshold sensitivity to cost assumptions")
plt.tight_layout()
plt.savefig("../reports/figures/threshold_sensitivity.png", dpi=120)
plt.show()


**Would different customer segments justify different thresholds?**
Plausibly yes — e.g. high-value long-tenure customers might warrant a lower
(more sensitive) threshold since a false negative is costlier in lost
lifetime value. Advanced Challenge C (segment-specific thresholds) explores
this formally; it requires minimum sample-size rules per segment and carries
real fairness/operational risk if applied naively (e.g. systematically
under-serving a demographic group), so it is treated as an EXPERT-level
extension rather than a default.


## 3. Interpretation: permutation importance

In [ ]:
perm_importance = permutation_importance_report(final_pipeline, X_val, y_val, n_repeats=10, random_state=RANDOM_SEED)
perm_importance.head(15)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
top = perm_importance.head(12).sort_values("importance_mean")
ax.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"], color="#4C72B0")
ax.set_title("Permutation importance (validation set, ROC-AUC drop)")
plt.tight_layout()
plt.savefig("../reports/figures/permutation_importance.png", dpi=120)
plt.show()


## 4. SHAP explanations (falls back gracefully if `shap` is not installed)

In [ ]:
shap_values = shap_explanation(final_pipeline, X_val.sample(min(200, len(X_val)), random_state=RANDOM_SEED))
if shap_values is not None:
    import shap
    shap.summary_plot(shap_values, show=False)
    plt.tight_layout()
    plt.savefig("../reports/figures/shap_summary.png", dpi=120)
    plt.show()
else:
    print("SHAP not available in this environment — permutation importance above is the "
          "model-agnostic fallback. Install `pip install shap` for individual-prediction "
          "waterfall explanations.")


## 5. Individual explanations: true positives, false positives, false negatives

In [ ]:
val_with_preds = val_df.copy()
val_with_preds["probability"] = val_proba
val_with_preds["prediction"] = (val_proba >= optimal_threshold).astype(int)

true_positives = val_with_preds[(val_with_preds[TARGET_COLUMN] == 1) & (val_with_preds["prediction"] == 1)]
false_positives = val_with_preds[(val_with_preds[TARGET_COLUMN] == 0) & (val_with_preds["prediction"] == 1)]
false_negatives = val_with_preds[(val_with_preds[TARGET_COLUMN] == 1) & (val_with_preds["prediction"] == 0)]

print("--- 3 True Positives (correctly flagged churners) ---")
print(true_positives[["customerID", "Contract", "tenure", "MonthlyCharges", "probability"]].head(3).to_string(index=False))

print("\n--- 3 False Positives (flagged but did not churn) ---")
print(false_positives[["customerID", "Contract", "tenure", "MonthlyCharges", "probability"]].head(3).to_string(index=False))

print("\n--- 3 False Negatives (missed churners) ---")
print(false_negatives[["customerID", "Contract", "tenure", "MonthlyCharges", "probability"]].head(3).to_string(index=False))


**Which features consistently drive churn risk?** Cross-reference the
permutation-importance ranking with the coefficients from notebook 03 —
contract type, tenure, and internet/payment-method features should appear in
both. **Which explanations reveal model weakness rather than customer
behavior?** False negatives clustered among long-tenure customers with a
sudden late-life churn (a pattern the model under-weights because most
churn in training data is early-tenure) would indicate a genuine blind spot,
not just irreducible noise.


## 6. Reliability: subgroup metrics and bootstrap confidence intervals

In [ ]:
val_with_preds["actual_churn"] = val_with_preds[TARGET_COLUMN]
sub_report = subgroup_metrics(
    val_with_preds, y_true_col=TARGET_COLUMN, y_pred_col="prediction",
    y_proba_col="probability", group_col="Contract", min_group_size=20,
)
sub_report


In [ ]:
point, lo, hi = bootstrap_metric_ci(y_val.values, val_proba, n_bootstrap=500, random_state=RANDOM_SEED)
print(f"ROC-AUC = {point:.4f}  (95% bootstrap CI: [{lo:.4f}, {hi:.4f}])")


Report metric differences relative to the width of this confidence
interval — e.g. a 0.01 ROC-AUC improvement from tuning is not meaningfully
different from zero if the CI half-width is 0.03.

## 7. Final evaluation — TEST SET, used exactly once


In [ ]:
test_proba = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = evaluate_at_threshold(
    y_test, test_proba, threshold=optimal_threshold,
    measure_latency_model=final_pipeline, X_for_latency=X_test.iloc[:200],
)
print("FINAL TEST METRICS (frozen threshold, evaluated once):")
print(test_metrics)


In [ ]:
import json
with open("../reports/final_test_metrics.json", "w") as f:
    json.dump(test_metrics.as_dict(), f, indent=2)
print("Saved reports/final_test_metrics.json")


## 8. Final risk report — `predictions.csv`

In [ ]:
artifact = ModelArtifact(
    pipeline=final_pipeline, threshold=optimal_threshold,
    model_version="logistic_regression-notebook-demo",
    trained_at=pd.Timestamp.now(tz="UTC").isoformat(),
    metrics_at_threshold=test_metrics.as_dict(),
)
risk_report = generate_final_risk_report(
    artifact, X_test, y_test.values, test_df[ID_COLUMN], output_path="../predictions.csv",
)
risk_report.head(10)


## 9. Executive summary (fill in with real-data results before publishing)

**Recommendation:** deploy the logistic-regression pipeline with the frozen
threshold shown above. It offers the best combination of validated ROC-AUC,
interpretability (coefficients map directly to business levers like contract
type and tech support), and lowest expected validation-set cost under the
stated \$500 / \$30 cost assumption.

**Key limitations:**
- Cost assumptions (\$500 false negative, \$30 false positive) are
  illustrative; replace with the business's actual retention-offer cost and
  estimated customer lifetime value before acting on this threshold in
  production.
- The model has not been validated on real Kaggle data if this notebook was
  run against the synthetic fallback — re-run end-to-end against the actual
  Telco Customer Churn CSV before drawing production conclusions.
- Observed associations (e.g. contract type, payment method) are not proven
  causal; retention interventions based on them should be tested, not assumed
  effective.

**Next steps / what data would improve the next version:** customer support
ticket history, NPS/satisfaction survey scores, and competitor promotional
activity would likely add predictive signal beyond what's available in this
billing-and-subscription-only dataset.

## Exit criterion
✅ Chosen model, frozen threshold, calibration check, global + individual
interpretation, subgroup reliability check with sample-size caveats,
bootstrap uncertainty, and a single, final, honest test-set evaluation.
